In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn import datasets
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.preprocessing import TargetEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

X_train = pd.read_parquet("../data/processed/X_train.parquet")
X_test = pd.read_parquet("../data/processed/X_test.parquet")
X_val = pd.read_parquet("../data/processed/X_val.parquet")

y_train = pd.read_parquet("../data/processed/y_train.parquet")["price"]
y_val = pd.read_parquet("../data/processed/y_val.parquet")["price"]
y_test = pd.read_parquet("../data/processed/y_test.parquet")["price"]

In [2]:
X_train.head(10)

,city,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,...,salvage,transmission,trim_name,wheel_system,year,mileage_missing,horsepower_missing,new_mileage_conflict,used_owner_count_missing,Condition_reported
1201912,Orlando,V6,True,Gasoline,False,283.0,False,Dodge,144084.0,Grand Caravan,...,False,A,SE FWD,FWD,2013,0,0,False,False,True
770704,Princeton,V6,False,Gasoline,False,270.0,False,Toyota,65165.0,Highlander,...,False,A,XLE V6 AWD,AWD,2015,0,0,False,False,True
1416224,Melbourne,V8 Flex Fuel Vehicle,False,Flex Fuel Vehicle,False,355.0,False,Chevrolet,58107.0,Tahoe,...,False,A,LT RWD,4X2,2017,0,0,False,False,True
250456,Catskill,V6,Not Reported,Gasoline,Not Reported,450.0,True,Ford,5.0,F-150,...,Not Reported,A,Limited SuperCrew 4WD,4WD,2020,0,0,False,False,False
2545573,Riverside,I4,False,Gasoline,False,150.0,False,Volkswagen,33709.0,Jetta,...,False,A,1.4T S FWD,FWD,2017,0,0,False,False,True
2149151,Dallas,V10,False,Gasoline,True,500.0,False,Dodge,19789.0,Viper,...,True,M,SRT10 Roadster RWD,RWD,2005,0,0,False,False,True
1904730,Tahlequah,V8,False,Gasoline,False,370.0,False,Dodge,21422.0,Charger,...,False,A,Daytona RWD,RWD,2017,0,0,False,False,True
660149,El Paso,V8,Not Reported,Gasoline,Not Reported,420.0,True,GMC,7.0,Sierra 1500,...,Not Reported,A,AT4 Crew Cab 4WD,4WD,2020,0,0,False,False,False
1834720,Washington,I4,Not Reported,Gasoline,Not Reported,270.0,True,Jeep,13.0,Cherokee,...,Not Reported,A,Altitude 4WD,4WD,2020,0,0,False,False,False
1024170,Kennesaw,V8 Flex Fuel Vehicle,False,Flex Fuel Vehicle,False,360.0,False,Chevrolet,23621.0,Silverado 2500HD,...,False,A,LT Double Cab 4WD,4WD,2015,0,0,False,False,True


In [3]:
X_train.shape

(2130204, 21)

In [4]:
X_val.shape

(266271, 21)

In [5]:
X_test.shape


(266276, 21)

In [6]:
categorical_cols = ['city', 'make_name', 'model_name','trim_name','engine_type','frame_damaged','fuel_type','has_accidents','salvage','transmission','wheel_system'] 

categoric_transformer= ('cat', OneHotEncoder(handle_unknown = 'ignore'), categorical_cols)


numerical_cols = ['horsepower', 'mileage', 'owner_count', 'year']
numeric_transformer = ('num', StandardScaler(), numerical_cols)



boolean_cols =['is_new',
 'mileage_missing',
 'horsepower_missing',
 'Condition_reported',
 'new_mileage_conflict',
 'used_owner_count_missing'] 
boolean_transformer = ("bool", "passthrough", boolean_cols)

preprocessor = ColumnTransformer(
    transformers=[
        numeric_transformer,
        categoric_transformer,
        boolean_transformer
    ]
)

In [7]:
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

In [8]:
X_train_processed.shape

(2130204, 14590)

In [9]:
X_val_processed.shape

(266271, 14590)

In [10]:
X_test_processed.shape

(266276, 14590)

In [11]:
lr_model = LinearRegression()
lr_model.fit(X_train_processed, y_train)
y_pred = lr_model.predict(X_val_processed)
y_prediction = lr_model.predict(X_train_processed)
t_prediction = lr_model.predict(X_test_processed)

In [12]:
mae = mean_absolute_error(y_val,y_pred)
val_rmse = np.sqrt(mean_squared_error(y_val,y_pred))
train_rmse = np.sqrt(mean_squared_error(y_train, y_prediction))
test_rmse = np.sqrt(mean_squared_error(y_test, t_prediction))
r2 = r2_score(y_val, y_pred)


print("MAE:", mae)
print("Val_rmse:", val_rmse)
print("train_rmse:",train_rmse)
print("test_rmse:",test_rmse)
print("r2:", r2)


MAE: 3023.446104768251
Val_rmse: 5457.046818417853
train_rmse: 6584.1260340418885
test_rmse: 8034.128195254633
r2: 0.9193459868619616


In [13]:
X_train.head(10)

,city,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,...,salvage,transmission,trim_name,wheel_system,year,mileage_missing,horsepower_missing,new_mileage_conflict,used_owner_count_missing,Condition_reported
1201912,Orlando,V6,True,Gasoline,False,283.0,False,Dodge,144084.0,Grand Caravan,...,False,A,SE FWD,FWD,2013,0,0,False,False,True
770704,Princeton,V6,False,Gasoline,False,270.0,False,Toyota,65165.0,Highlander,...,False,A,XLE V6 AWD,AWD,2015,0,0,False,False,True
1416224,Melbourne,V8 Flex Fuel Vehicle,False,Flex Fuel Vehicle,False,355.0,False,Chevrolet,58107.0,Tahoe,...,False,A,LT RWD,4X2,2017,0,0,False,False,True
250456,Catskill,V6,Not Reported,Gasoline,Not Reported,450.0,True,Ford,5.0,F-150,...,Not Reported,A,Limited SuperCrew 4WD,4WD,2020,0,0,False,False,False
2545573,Riverside,I4,False,Gasoline,False,150.0,False,Volkswagen,33709.0,Jetta,...,False,A,1.4T S FWD,FWD,2017,0,0,False,False,True
2149151,Dallas,V10,False,Gasoline,True,500.0,False,Dodge,19789.0,Viper,...,True,M,SRT10 Roadster RWD,RWD,2005,0,0,False,False,True
1904730,Tahlequah,V8,False,Gasoline,False,370.0,False,Dodge,21422.0,Charger,...,False,A,Daytona RWD,RWD,2017,0,0,False,False,True
660149,El Paso,V8,Not Reported,Gasoline,Not Reported,420.0,True,GMC,7.0,Sierra 1500,...,Not Reported,A,AT4 Crew Cab 4WD,4WD,2020,0,0,False,False,False
1834720,Washington,I4,Not Reported,Gasoline,Not Reported,270.0,True,Jeep,13.0,Cherokee,...,Not Reported,A,Altitude 4WD,4WD,2020,0,0,False,False,False
1024170,Kennesaw,V8 Flex Fuel Vehicle,False,Flex Fuel Vehicle,False,360.0,False,Chevrolet,23621.0,Silverado 2500HD,...,False,A,LT Double Cab 4WD,4WD,2015,0,0,False,False,True


In [14]:
val_hashes = pd.util.hash_pandas_object(X_val, index=False)
train_hashes = pd.util.hash_pandas_object(X_train, index=False)

In [15]:
val_group = val_hashes.groupby(val_hashes).groups
train_group = train_hashes.groupby(train_hashes).groups

In [16]:
mask = np.isin(val_hashes, train_hashes)

In [17]:
matched_val = X_val.loc[mask]

In [18]:
len(matched_val)
len(X_val)

266271

In [19]:
len(matched_val) / len(X_val) * 100

19.202992440032897

In [20]:
train_lookup = pd.DataFrame({
    "hash": train_hashes,
    "train_price": y_train
})

val_lookup = pd.DataFrame({
    "hash": val_hashes,
    "val_price": y_val
})

In [21]:
matched_prices = val_lookup.merge(
    train_lookup,
    on="hash",
    how="inner"
)

In [22]:
different_prices = matched_prices[
    matched_prices["val_price"] != matched_prices["train_price"]
]

In [23]:
len(different_prices)

171752

In [24]:
different_prices.head(20)

,hash,val_price,train_price
0,17846470405846719530,59286.0,54255.0
1,6733417902417196245,21178.0,24913.0
2,6733417902417196245,21178.0,25741.0
3,6733417902417196245,21178.0,24366.0
4,6733417902417196245,21178.0,24090.0
5,6733417902417196245,21178.0,26288.0
6,4212990437395690760,26181.0,23004.0
7,4212990437395690760,26181.0,23827.0
8,4212990437395690760,26181.0,23472.0
9,4212990437395690760,26181.0,23280.0


In [25]:
different_prices.describe()

,hash,val_price,train_price
count,1.717520e+05,171752.000000,1.717520e+05
mean,9.183646e+18,38657.968544,3.863267e+04
std,5.323869e+18,16832.940019,1.660666e+04
min,5.817745e+14,9270.000000,9.285000e+03
25%,4.649668e+18,26257.000000,2.624900e+04
50%,9.098538e+18,36381.000000,3.638500e+04
75%,1.385261e+19,47466.000000,4.753500e+04
max,1.844541e+19,425460.000000,1.116711e+06


In [26]:
different_prices["price_difference"] = (
    different_prices["val_price"] - different_prices["train_price"]
).abs()

different_prices.sort_values(
    "price_difference",
    ascending=False
).head(30)

,hash,val_price,train_price,price_difference
123399,16468349563739211745,57500.0,1116711.0,1059211.0
32745,16468349563739211745,59475.0,1116711.0,1057236.0
92438,6713839979747926329,425460.0,32368.0,393092.0
92428,6713839979747926329,425460.0,32382.0,393078.0
92432,6713839979747926329,425460.0,32828.0,392632.0
44333,6713839979747926329,424060.0,32368.0,391692.0
44323,6713839979747926329,424060.0,32382.0,391678.0
44327,6713839979747926329,424060.0,32828.0,391232.0
92437,6713839979747926329,425460.0,35386.0,390074.0
92425,6713839979747926329,425460.0,35846.0,389614.0


In [58]:
suspect_hash = 6713839979747926329

val_rows = X_val.loc[val_hashes == suspect_hash].copy()
val_rows["price"] = y_val.loc[val_rows.index]

train_rows = X_train.loc[train_hashes == suspect_hash].copy()
train_rows["price"] = y_train.loc[train_rows.index]

In [59]:
val_rows

,city,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,...,transmission,trim_name,wheel_system,year,mileage_missing,horsepower_missing,new_mileage_conflict,used_owner_count_missing,Condition_reported,price
908527,Charlotte,Unknown,Not Reported,Unknown,Not Reported,345.0,True,GMC,5.0,Acadia,...,A,Unknown,Unknown,2021,0,1,False,False,False,424060.0
908532,Charlotte,Unknown,Not Reported,Unknown,Not Reported,345.0,True,GMC,5.0,Acadia,...,A,Unknown,Unknown,2021,0,1,False,False,False,425460.0
907408,Charlotte,Unknown,Not Reported,Unknown,Not Reported,345.0,True,GMC,5.0,Acadia,...,A,Unknown,Unknown,2021,0,1,False,False,False,42867.0
907359,Charlotte,Unknown,Not Reported,Unknown,Not Reported,345.0,True,GMC,5.0,Acadia,...,A,Unknown,Unknown,2021,0,1,False,False,False,52345.0


In [60]:
train_rows.head(20)

,city,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,...,transmission,trim_name,wheel_system,year,mileage_missing,horsepower_missing,new_mileage_conflict,used_owner_count_missing,Condition_reported,price
907663,Charlotte,Unknown,Not Reported,Unknown,Not Reported,345.0,True,GMC,5.0,Acadia,...,A,Unknown,Unknown,2021,0,1,False,False,False,48727.0
906951,Charlotte,Unknown,Not Reported,Unknown,Not Reported,345.0,True,GMC,5.0,Acadia,...,A,Unknown,Unknown,2021,0,1,False,False,False,35846.0
907648,Charlotte,Unknown,Not Reported,Unknown,Not Reported,345.0,True,GMC,5.0,Acadia,...,A,Unknown,Unknown,2021,0,1,False,False,False,48169.0
907369,Charlotte,Unknown,Not Reported,Unknown,Not Reported,345.0,True,GMC,5.0,Acadia,...,A,Unknown,Unknown,2021,0,1,False,False,False,46421.0
906836,Charlotte,Unknown,Not Reported,Unknown,Not Reported,345.0,True,GMC,5.0,Acadia,...,A,Unknown,Unknown,2021,0,1,False,False,False,32382.0
907875,Charlotte,Unknown,Not Reported,Unknown,Not Reported,345.0,True,GMC,5.0,Acadia,...,A,Unknown,Unknown,2021,0,1,False,False,False,50154.0
907354,Charlotte,Unknown,Not Reported,Unknown,Not Reported,345.0,True,GMC,5.0,Acadia,...,A,Unknown,Unknown,2021,0,1,False,False,False,51991.0
907289,Charlotte,Unknown,Not Reported,Unknown,Not Reported,345.0,True,GMC,5.0,Acadia,...,A,Unknown,Unknown,2021,0,1,False,False,False,51787.0
906916,Charlotte,Unknown,Not Reported,Unknown,Not Reported,345.0,True,GMC,5.0,Acadia,...,A,Unknown,Unknown,2021,0,1,False,False,False,32828.0
907320,Charlotte,Unknown,Not Reported,Unknown,Not Reported,345.0,True,GMC,5.0,Acadia,...,A,Unknown,Unknown,2021,0,1,False,False,False,42407.0


Drama in the shhaaat

Investigated why train RMSE > validation RMSE.

Turns out my earlier Mercedes/BMW cleanup wasn't comprehensive. I removed the symptoms, not the root cause.

Found more corrupted prices hiding in identical-feature groups, including a GMC Acadia listed at ~$400k 💀

Next: systematically clean the root issue → retrain → re-evaluate.

In [70]:
vehicle_lookup = X_train.copy()
vehicle_lookup["hash"] = train_hashes.values

In [71]:
vehicle_lookup = vehicle_lookup[
    ["hash", "make_name", "model_name", "year", "trim_name", "mileage"]
]

In [72]:
vehicle_lookup = vehicle_lookup.drop_duplicates(subset="hash")

In [31]:
val_prices = val_lookup.rename(columns={"val_price": "price"})
train_prices = train_lookup.rename(columns={"train_price": "price"})

all_prices = pd.concat([train_prices, val_prices], ignore_index=True)

In [32]:
price_groups = all_prices.groupby("hash")

In [35]:
price_stats = price_groups["price"].quantile([0.25, 0.75]).unstack()

In [36]:
price_stats.columns = ["Q1", "Q3"]
price_stats["IQR"] = price_stats["Q3"] - price_stats["Q1"]

In [38]:
price_stats["lower_bound"] = price_stats["Q1"] - 1.5 * price_stats["IQR"]
price_stats["upper_bound"] = price_stats["Q3"] + 1.5 * price_stats["IQR"]

In [39]:
prices_with_bounds = all_prices.merge(
    price_stats,
    on="hash",
    how="left"
)

In [40]:
price_stats.head()

,Q1,Q3,IQR,lower_bound,upper_bound
hash,,,,,
1187703273057,41912.0,41912.0,0.0,41912.0,41912.0
13228759559581,26125.0,26125.0,0.0,26125.0,26125.0
22701073440238,23395.0,23395.0,0.0,23395.0,23395.0
28112847962006,24492.0,24492.0,0.0,24492.0,24492.0
28186126298156,39988.0,39988.0,0.0,39988.0,39988.0


In [41]:
price_stats = price_stats.reset_index()

In [42]:
price_stats.head()

,hash,Q1,Q3,IQR,lower_bound,upper_bound
0,1187703273057,41912.0,41912.0,0.0,41912.0,41912.0
1,13228759559581,26125.0,26125.0,0.0,26125.0,26125.0
2,22701073440238,23395.0,23395.0,0.0,23395.0,23395.0
3,28112847962006,24492.0,24492.0,0.0,24492.0,24492.0
4,28186126298156,39988.0,39988.0,0.0,39988.0,39988.0


In [43]:
prices_with_bounds = all_prices.merge(
    price_stats,
    on="hash",
    how="left"
)

In [44]:
outlier_mask = (
    (prices_with_bounds["price"] < prices_with_bounds["lower_bound"]) |
    (prices_with_bounds["price"] > prices_with_bounds["upper_bound"])
)

In [45]:
price_outliers = prices_with_bounds.loc[outlier_mask]

In [46]:
price_outliers

,hash,price,Q1,Q3,IQR,lower_bound,upper_bound
360,11492454641014865074,37129.0,33689.00,34660.00,971.00,32232.500,36116.500
382,18354971751878231366,30741.0,31359.75,31666.00,306.25,30900.375,32125.375
506,15939350068581148109,64508.0,60046.00,60994.00,948.00,58624.000,62416.000
988,1481733677770707769,70395.0,70631.25,70785.00,153.75,70400.625,71015.625
1309,15400783789799948445,31028.0,31164.50,31248.50,84.00,31038.500,31374.500
...,...,...,...,...,...,...,...
2395270,462773034933269947,22439.0,19923.50,20907.00,983.50,18448.250,22382.250
2395321,4176720475846868057,25215.0,26245.00,26275.00,30.00,26200.000,26320.000
2395337,14762203832802342259,45453.0,39821.50,41947.00,2125.50,36633.250,45135.250
2395564,14527353990557763932,34560.0,34762.50,34800.00,37.50,34706.250,34856.250


In [47]:
median_prices = price_groups["price"].median()

In [48]:
median_prices.head()

hash
1187703273057     41912.0
13228759559581    26125.0
22701073440238    23395.0
28112847962006    24492.0
28186126298156    39988.0
Name: price, dtype: float64

In [49]:
median_prices = median_prices.reset_index(name="median_price")

In [50]:
prices_with_bounds = prices_with_bounds.merge(
    median_prices,
    on="hash",
    how="left"
)

In [ ]:
prices_with_bounds["price_ratio"] = (
    prices_with_bounds["price"] / prices_with_bounds["median_price"]
)

In [52]:
prices_with_bounds["price_ratio"]

0          1.0
1          1.0
2          1.0
3          1.0
4          1.0
          ... 
2396470    1.0
2396471    1.0
2396472    1.0
2396473    1.0
2396474    1.0
Name: price_ratio, Length: 2396475, dtype: float64

In [55]:
prices_with_bounds["price_ratio"].sort_values(ascending = False).head(20)

41194      18.875318
9447       10.112927
2274359     8.832652
2199403     8.803587
1834262     8.756254
2070733     8.072827
1789711     3.755831
747564      3.151632
1852629     2.890950
1940851     2.527994
645841      2.303703
2193739     2.298284
688861      2.226770
2332802     2.218362
358795      2.216688
1029122     2.200260
780581      2.102827
2124651     2.090569
1181871     2.070429
1601663     2.035143
Name: price_ratio, dtype: float64

In [66]:
sus_hash = 16468349563739211745
val_rows = X_val.loc[val_hashes == sus_hash].copy()
val_rows["price"] = y_val.loc[val_rows.index]

train_rows = X_train.loc[train_hashes == sus_hash].copy()
train_rows["price"] = y_train.loc[train_rows.index]

In [67]:
train_rows

,city,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,...,transmission,trim_name,wheel_system,year,mileage_missing,horsepower_missing,new_mileage_conflict,used_owner_count_missing,Condition_reported,price
2537100,Laguna Niguel,I4,Not Reported,Gasoline,Not Reported,180.0,True,Mercedes-Benz,1.0,E-Class,...,A,E 350 Sedan RWD,Unknown,2021,0,1,False,False,False,1116711.0
2535958,Laguna Niguel,I4,Not Reported,Gasoline,Not Reported,180.0,True,Mercedes-Benz,1.0,E-Class,...,A,E 350 Sedan RWD,Unknown,2021,0,1,False,False,False,58850.0


In [68]:
val_rows

,city,engine_type,frame_damaged,fuel_type,has_accidents,horsepower,is_new,make_name,mileage,model_name,...,transmission,trim_name,wheel_system,year,mileage_missing,horsepower_missing,new_mileage_conflict,used_owner_count_missing,Condition_reported,price
2536065,Laguna Niguel,I4,Not Reported,Gasoline,Not Reported,180.0,True,Mercedes-Benz,1.0,E-Class,...,A,E 350 Sedan RWD,Unknown,2021,0,1,False,False,False,59475.0
2535528,Laguna Niguel,I4,Not Reported,Gasoline,Not Reported,180.0,True,Mercedes-Benz,1.0,E-Class,...,A,E 350 Sedan RWD,Unknown,2021,0,1,False,False,False,57500.0


In [73]:
prices_with_vehicle = prices_with_bounds.merge(
    vehicle_lookup,
    on="hash",
    how="left"
)

In [80]:
prices_with_vehicle.sort_values(
    "price_ratio",
    ascending=False
)[
    [
        "make_name",
        "model_name",
        "year",
        "trim_name",
        "mileage",
        "price",
        "median_price",
        "price_ratio"
    ]
]

,make_name,model_name,year,trim_name,mileage,price,median_price,price_ratio
41194,Mercedes-Benz,E-Class,2021.0,E 350 Sedan RWD,1.0,1116711.0,59162.5,18.875318
9447,Jeep,Compass,2021.0,Unknown,6.0,320150.0,31657.5,10.112927
2274359,GMC,Acadia,2021.0,Unknown,5.0,425460.0,48169.0,8.832652
2199403,GMC,Acadia,2021.0,Unknown,5.0,424060.0,48169.0,8.803587
1834262,GMC,Acadia,2021.0,Unknown,5.0,421780.0,48169.0,8.756254
...,...,...,...,...,...,...,...,...
1496518,Cadillac,Series 62,1949.0,Unknown,41308.0,9495.0,39745.0,0.238898
1563674,Jeep,Gladiator,2021.0,Unknown,5.0,61470.0,278886.0,0.220413
2266472,Ford,Transit Cargo,2020.0,Unknown,13.0,44370.0,235410.0,0.188480
964528,Chevrolet,Blazer,2021.0,Unknown,3.0,48395.0,270347.5,0.179010


In [81]:
len(all_prices)
len(all_prices["hash"].unique())

2073746

In [82]:
all_prices["hash"].value_counts().head()

hash
2470524655507885768     90
13498212496571669789    81
8068587227830895646     77
14588591188894768402    64
6489687123355713297     62
Name: count, dtype: int64

hashed X_train and X_val
used the hashes to identify rows with identical feature values
attached y_train / y_val prices to those hashes
found groups where the same X has different prices
confirmed obvious corruption like:
2021 Mercedes E-Class around $1.1M vs normal ~$58–59k
2021 GMC Acadia around $389k–$425k vs normal ~$32–52k
Jeep Compass around $320k vs median ~$31k
built group-level stats:
Q1
Q3
IQR
lower/upper IQR bounds
median price
calculated price_ratio = price / median_price
learned that IQR alone is too aggressive because it flags normal market variation
learned that price ratio alone is also too crude
decided the cleanup rule should combine both:
row is outside the group’s IQR bounds
AND
price ratio is extreme

Important correction we caught at the end:

We accidentally calculated these stats across all train + val hashes, including singleton hashes. That’s why the table exploded to millions of rows.